# Train3 단독 분석 — KSPHM-KIMM 2026

> **역할 분담**: 본 노트북은 4개 Train 중 **Train3** 한 세트만 깊이 있게 분석합니다 (데이터 분석/EDA 범위만, 예측 모델링은 별도 단계).
> 전체 4 Train 비교는 [00_walkthrough.ipynb](00_walkthrough.ipynb)를 참고. Train2 비교 분석은 [01_train2_analysis.ipynb](01_train2_analysis.ipynb).

## Train3 한눈에 (객관 측정)

| 항목 | 값 |
|---|---|
| 시험 시간 | 14.8 h (Operation 5,321행 × 10초) |
| 진동 파일 수 | 89 (10분 주기 × 1분 측정) |
| 종료 토크 | -20.8 Nm (자동 정지 트리거 -20 Nm) |
| TC SP Front max | **191.9 °C** ← 더 뜨거운 쪽 (200°C 트리거 근접) |
| TC SP Rear max | 124.3 °C |
| 시험 정지 원인 | 토크 트리거 (Front 200°C 근접하지만 미도달) |
| 가장 큰 envelope 비율 | **CH1 BSF 1x = 13.0×** |
| 특이 사항 1 | **모든 4채널 kurtogram BP 정상** (fallback 0/4) — Train2와 정반대 |
| 특이 사항 2 | RMS·BPFx 변화가 **Front 채널(CH1·CH2)에 집중** — Front 베어링 열화 |

## 분석 흐름 (데이터 분석만)

1. 데이터 인벤토리 + Operation 시그널
2. 4채널 시간영역 트렌드 (RMS/Kurt/CF)
3. Early vs Late 상세 비교 (waveform + Welch PSD)
4. Kurtogram 기반 BP + Envelope 분석 + 진단 heatmap
5. BPFx 라인의 late/early 비율 (객관 측정)
6. 종합 발견 — 객관 수치 정리

---


## 0. 환경 설정

In [ ]:
import sys
from pathlib import Path

ROOT = Path('c:/Users/User/WorkSpace/data_challenge')
for p in (ROOT, ROOT / 'utils'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt, hilbert
import scipy.stats as ss

from src.io_tdms import FS, CHANNEL_NAMES, load_tdms_file, tdms_to_array
from src.operation import load_operation, list_vibration_files, align_to_vibration

TR = 3  # focus on Train3 only
FEAT_DIR = ROOT / 'outputs' / 'features_utils'
print(f'분석 대상: Train{TR}')
print(f'numpy/pandas: {np.__version__} / {pd.__version__}')

---

## 1. Train3 데이터 인벤토리


In [ ]:
files = list_vibration_files(TR)
op = load_operation(TR)

print(f'진동 파일 수: {len(files)}  ({files[0].name} ~ {files[-1].name})')
print(f'TDMS 1개 크기: {files[0].stat().st_size/1024/1024:.2f} MB')
print(f'Operation 행수: {len(op):,}  (10초 간격, 0.1 Hz)')
print(f'시험 시간: {op["Time[sec]"].iloc[-1]/3600:.2f} h')
print()
print('Operation 컬럼별 통계:')
op[['Torque[Nm]', 'Motor speed[rpm]', 'TC SP Front', 'TC SP Rear']].describe().round(2)

### 1.1 Operation 시그널 — RPM / Torque / Temp 시계열

운전 조건의 변화와 정지 트리거를 시각화. RPM step 패턴과 토크 spike, **Front 온도의 200°C 근접** 추세를 확인.


In [ ]:
t_h = op['Time[sec]'].to_numpy() / 3600.0
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)

axes[0].plot(t_h, op['Motor speed[rpm]'], color='tab:blue', lw=0.6)
axes[0].set_ylabel('RPM')
axes[0].grid(alpha=0.3)
axes[0].set_title(f'Train{TR} — Operation 시그널 ({op["Time[sec]"].iloc[-1]/3600:.1f} h)')

axes[1].plot(t_h, op['Torque[Nm]'], color='tab:purple', lw=0.5)
axes[1].axhline(-20, color='red', ls='--', lw=0.7, label='-20 Nm 정지 트리거')
axes[1].set_ylabel('Torque [Nm]')
axes[1].grid(alpha=0.3); axes[1].legend(fontsize=9)

axes[2].plot(t_h, op['TC SP Front'], color='tab:orange', lw=0.6, label='Front')
axes[2].plot(t_h, op['TC SP Rear'],  color='tab:green',  lw=0.6, label='Rear')
axes[2].axhline(200, color='red', ls='--', lw=0.7, label='200°C 트리거')
axes[2].set_ylabel('Temp [°C]')
axes[2].set_xlabel('Time [hours]')
axes[2].grid(alpha=0.3); axes[2].legend(fontsize=9)

fig.tight_layout()
plt.show()

print(f'\nFront max: {op["TC SP Front"].max():.1f}°C   Rear max: {op["TC SP Rear"].max():.1f}°C')
print(f'더 뜨거운 쪽: {"Rear" if op["TC SP Rear"].max() > op["TC SP Front"].max() else "Front"}')
print(f'종료 토크: {op["Torque[Nm]"].iloc[-1]:.2f} Nm')

> **읽기**
> - **RPM**: 700 ↔ 950 약 1시간 주기 교번 (Train2와 유사)
> - **Torque**: 평소 -2~-5 Nm 부근 → 마지막 시점에 -20.8 Nm spike → 자동 정지
> - **Temp**: **Front가 시간 따라 단조 상승해 191.9°C** (200°C 트리거 거의 근접) / Rear는 124°C 부근에 머묾 → **Front 베어링 마찰 증가가 가장 강한 객관 신호** (Train2와 정반대 패턴)
> - 200°C 트리거는 도달 X — 정지는 토크 -20 Nm 트리거


---

## 2. 4채널 시간영역 트렌드 (RMS / Peak / Kurtosis)

89개 파일 × 4채널 = 356개 1분 세그먼트의 시간영역 통계 추세. `outputs/features_utils/train3.parquet`에 캐시된 피처를 사용.


In [ ]:
df_t3 = pd.read_parquet(FEAT_DIR / f'train{TR}.parquet').sort_values('file_idx').reset_index(drop=True)
print(f'Feature shape: {df_t3.shape}')
print(f'Feature columns include: {[c for c in df_t3.columns if "_rms" == c[-4:]][:4]} ...')
df_t3.head(3)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
colors = {'CH1': 'tab:blue', 'CH2': 'tab:orange', 'CH3': 'tab:green', 'CH4': 'tab:red'}
t_h = df_t3['t_start_sec'] / 3600

for ch in CHANNEL_NAMES:
    axes[0].plot(t_h, df_t3[f'{ch}_rms'],  color=colors[ch], lw=1.2, label=ch)
    axes[1].plot(t_h, df_t3[f'{ch}_kurt'], color=colors[ch], lw=1.0, label=ch)
    axes[2].plot(t_h, df_t3[f'{ch}_cf'],   color=colors[ch], lw=1.0, label=ch)

axes[0].set_ylabel('RMS'); axes[0].grid(alpha=0.3); axes[0].legend(fontsize=9, ncol=4)
axes[0].set_title(f'Train{TR} — 4-channel time-domain trends')
axes[1].set_ylabel('Kurtosis'); axes[1].grid(alpha=0.3)
axes[1].set_yscale('symlog', linthresh=10)  # CH4 has extreme spike
axes[2].set_ylabel('Crest Factor'); axes[2].grid(alpha=0.3)
axes[2].set_xlabel('Time [hours]')

fig.tight_layout()
plt.show()

# Numerical summary
print('\n--- 채널별 early/late 비교 (file 1 vs file 114) ---')
print(f'{"Ch":<4} {"RMS_e":>8} {"RMS_l":>8} {"RMS×":>6} {"Kurt_e":>8} {"Kurt_l":>10} {"CF_e":>6} {"CF_l":>6}')
for ch in CHANNEL_NAMES:
    rms_e, rms_l = df_t3[f'{ch}_rms'].iloc[0], df_t3[f'{ch}_rms'].iloc[-1]
    kt_e, kt_l = df_t3[f'{ch}_kurt'].iloc[0], df_t3[f'{ch}_kurt'].iloc[-1]
    cf_e, cf_l = df_t3[f'{ch}_cf'].iloc[0], df_t3[f'{ch}_cf'].iloc[-1]
    print(f'{ch:<4} {rms_e:>8.3f} {rms_l:>8.3f} {rms_l/rms_e:>5.1f}× {kt_e:>8.2f} {kt_l:>10.2f} {cf_e:>6.2f} {cf_l:>6.2f}')

> **객관 관찰**
>
> | Ch | RMS× (early→late) | Kurtosis_late |
> |---|---|---|
> | **CH1 (Front Vert.)** | **2.3×** | 6.5 (impulsive 증가) |
> | **CH2 (Front Axial)** | **3.1×** | 5.1 (impulsive 증가) |
> | CH3 (Rear Vert.) | 1.8× | 3.9 (정상 부근) |
> | CH4 (Rear Axial) | 1.7× | 3.2 (정상 부근) |
>
> - **Front 채널(CH1·CH2)에서 RMS 2-3× 성장 + Kurtosis 5~6.5로 상승** — Front 베어링에 결함 임펄스가 자라는 객관 증거
> - **Rear 채널(CH3·CH4)은 변화 미미** — Train2의 후면 우세 패턴과 정반대
> - Crest factor도 Front 채널에서 5.5 → 12+ 로 두 배 이상 증가 (peakiness 강화)
> - Train2 같은 극단적 단발 transient(Kurt > 100)는 없음 — Train3은 **점진적 균질 열화**


---

## 3. Early vs Late 상세 비교

첫 파일(`000001.tdms`)과 마지막 파일(`000089.tdms`)을 직접 로드해 1초 waveform과 FFT를 비교.

| | early | late |
|---|---|---|
| 파일 | 000001.tdms | 000089.tdms |
| 시각 | t = 0 ~ 60 s | t ≈ 14.7 h |
| RPM | (실측, 다음 셀에서 확인) | 약 716 |


In [ ]:
# Load endpoint TDMS files
early = tdms_to_array(load_tdms_file(files[0]))
late  = tdms_to_array(load_tdms_file(files[-1]))
print(f'early shape: {early.shape}, late shape: {late.shape}')

# Get RPM at endpoints
agg = align_to_vibration(op, len(files))
rpm_e = float(agg['rpm_mean'].iloc[0])
rpm_l = float(agg['rpm_mean'].iloc[-1])
print(f'early rpm: {rpm_e:.1f}, late rpm: {rpm_l:.1f}')

# Per-channel waveform 1 sec
n_show = FS  # 1 s
t_axis = np.arange(n_show) / FS

fig, axes = plt.subplots(4, 2, figsize=(14, 10), sharex='col')
for i, ch in enumerate(CHANNEL_NAMES):
    axes[i, 0].plot(t_axis, early[i, :n_show], color='tab:blue', lw=0.5, alpha=0.85)
    axes[i, 0].set_ylabel(f'{ch}\namp')
    axes[i, 0].grid(alpha=0.3)
    axes[i, 1].plot(t_axis, late[i, :n_show],  color='tab:red',  lw=0.5, alpha=0.85)
    axes[i, 1].grid(alpha=0.3)
    if i == 0:
        axes[i, 0].set_title(f'(a) Waveform — EARLY  rpm~{rpm_e:.0f}', fontsize=11)
        axes[i, 1].set_title(f'(b) Waveform — LATE  rpm~{rpm_l:.0f}', fontsize=11)
    if i == 3:
        axes[i, 0].set_xlabel('time [s]')
        axes[i, 1].set_xlabel('time [s]')

fig.suptitle(f'Train{TR} — first 1 second waveform per channel', y=1.0)
fig.tight_layout()
plt.show()

> **객관 관찰 (waveform)**
> - Early(파랑): 4채널 모두 진폭 작고 random noise 분포
> - Late(빨강): **CH1·CH2(Front)** 진폭 증가가 명확. CH2는 모터/기어 메싱 톤 위에 작은 임펄스가 얹혀 보임
> - CH3·CH4(Rear)는 late에서도 진폭 변화 미미 — 시간영역 트렌드(§2)와 일관


In [ ]:
# FFT comparison per channel — Welch PSD (averaged short windows)
# 60초 신호 raw FFT는 0.017 Hz 해상도라 시각적으로 noisy. Welch PSD는 4096-sample
# 세그먼트를 평균해 variance를 줄임. bearing diagnostics의 표준 spectrum view.
from scipy.signal import welch

fig, axes = plt.subplots(4, 1, figsize=(13, 11))
for i, ch in enumerate(CHANNEL_NAMES):
    for sig, lbl, color in [(early[i], 'early', 'tab:blue'),
                            (late[i],  'late',  'tab:red')]:
        f_w, P_w = welch(sig.astype(np.float64), fs=FS, nperseg=4096, noverlap=2048,
                         scaling='density')
        m = (f_w >= 5) & (f_w <= 13000)
        axes[i].semilogy(f_w[m], P_w[m], color=color, lw=0.8, alpha=0.85, label=lbl)
    axes[i].set_ylabel(f'{ch}\nPSD [V²/Hz]')
    axes[i].grid(alpha=0.3)
    axes[i].set_xlim(5, 13000)
    if i == 0:
        axes[i].legend(fontsize=9, loc='upper right')
        axes[i].set_title(f'Train{TR} — Welch PSD comparison (4096-sample segments averaged)')
    if i == 3: axes[i].set_xlabel('frequency [Hz]')
fig.tight_layout()
plt.show()

> **객관 관찰 (Welch PSD)**
>
> Welch PSD는 raw FFT(0.017 Hz/bin, 768K bins)보다 훨씬 부드럽고 (6.25 Hz/bin, 2049 bins) **신호 구조가 명확히 보임**:
>
> - **CH1 (Front Vert.)**: late에서 **9-10 kHz 근처 좁은 공진**이 새로 자람 → kurtogram이 [9.8-10.0] kHz BP를 자신 있게 선택할 carrier
> - **CH2 (Front Axial)**: late에서 **1-2 kHz 영역**과 **메싱 고조파 4·8·12 kHz**가 elevated → kurtogram BP는 [1.3-1.5] kHz로 결정
> - **CH3·CH4 (Rear)**: early/late 차이 매우 작음 — Rear는 거의 변화 없음
> - 모든 채널 공통: 4 kHz, 8 kHz, 12 kHz 부근의 좁은 라인 → 기어 메싱 또는 회전 고조파
>
> raw FFT보다 Welch PSD가 **bearing diagnostics의 표준** 인 이유: short-segment averaging으로 random variance를 줄여 *결정론적 결함 신호*가 잘 드러남.


---

## 4. Kurtogram 기반 노이즈 제거 + Envelope 분석

`fast_kurtogram`(Antoni 2007)을 4채널 각각에 적용해 임펄스가 가장 강한 BP 대역을 자동 검출.


In [ ]:
# Load pre-computed kurtogram bands
bands_csv = pd.read_csv(FEAT_DIR / 'selected_bands.csv')
bands_t3 = bands_csv[bands_csv.train == f'Train{TR}'].copy()
print(f'Train{TR} kurtogram 결과:')
print()
cols = ['channel', 'lo', 'hi', 'bw', 'fc', 'kmax', 'level', 'fallback']
print(bands_t3[cols].to_string(index=False))

> **Train3 BP 선정 결과**
>
> | Ch | BP 대역 | kmax | level | fallback | 의미 |
> |---|---|---|---|---|---|
> | **CH1** | **9.77-10.03 kHz (BW 267 Hz)** | **149.8** | 5.58 | False | 매우 선명한 좁은 공진 — Front Vert. 결함 carrier |
> | **CH2** | **1.33-1.53 kHz (BW 200 Hz)** | **171.1** | 6.00 | False | 저주파 좁은 공진 — Front Axial 결함 carrier |
> | CH3 | 6.17-6.70 kHz (BW 533 Hz) | 1.5 | 4.58 | False | kmax 작음 (impulse 약함) — 그래도 정상 BP 통과 |
> | CH4 | 11.53-11.73 kHz (BW 200 Hz) | 2.5 | 6.00 | False | kmax 작음 (impulse 약함) — 그래도 정상 BP 통과 |
>
> **모든 4채널에서 fallback 없이 좁은 BP가 선택됨** — Train2의 3/4 fallback과 정반대.
> 이는 Train3의 결함 신호가 **broad band saturation 없이 깔끔한 narrow resonance를 여기**시킨다는 뜻 — 점진적 결함 발달의 징표.


### 4a. Kurtogram 진단 — 왜 Train3은 모두 정상 BP인가

Train2(§01 노트북)에서 4채널 중 3채널이 fallback `[1, 10] kHz` 였던 것과 달리, Train3은 **4채널 모두 정상 BP**. 이를 직접 검증.

**fallback 트리거 규칙** ([src/features_utils.py:88-95](../src/features_utils.py#L88-L95)):
```python
if lvl < 1.0 or (hi - lo) < 150 or lo < 300 or hi > fs/2 - 100:
    return KURT_FALLBACK_BAND  # = [1000, 10000] Hz
```

| 조건 | 의미 |
|---|---|
| `lvl < 1.0` | kurtogram의 best level이 0 (전체 대역) → 좁은 좋은 대역 없음 |
| `(hi - lo) < 150` | BW 너무 좁아 leakage 우려 |
| `lo < 300` | 샤프트 영역과 겹침 |
| `hi > fs/2 - 100` | Nyquist 너무 가까움 |

**실측: Train3 4채널 kurtogram 출력 (selected_bands.csv)**

| Ch | lvl | lo | hi | kmax | 결과 |
|---|---|---|---|---|---|
| CH1 | 5.58 | 9767 | 10033 | 149.8 | **통과** (정상 BP) |
| CH2 | 6.00 | 1333 | 1533 | 171.1 | **통과** (정상 BP) |
| CH3 | 4.58 | 6167 | 6700 | 1.5 | **통과** (정상 BP) |
| CH4 | 6.00 | 11533 | 11733 | 2.5 | **통과** (정상 BP) |

CH1·CH2는 kmax > 100 으로 매우 선명한 결함 carrier 응답을 보이고, CH3·CH4는 kmax가 작지만 그래도 의미 있는 좁은 대역이 발견됨. Train2처럼 단발 거대 transient가 spectral kurtosis를 dominate하지 않기 때문에 4채널 모두 깔끔한 narrow BP 선정이 가능했음.

**확인 — kurtogram 직접 시각화**: 다음 셀에서 Train3 CH1(가장 강한 carrier)와 CH3(약한 kmax) 의 kurtogram heatmap을 그려서 narrow hot-spot이 정말 있는지 검증.


In [ ]:
# Kurtogram visualization on the EXACT file used by features_utils.py (000104.tdms = idx 103, ~92% of life)
import warnings
sys.path.insert(0, str(ROOT / 'utils'))
from kurtogram import fast_kurtogram

# features_utils picks use_idx = int(len(files) * 0.92) - 1 = 103 for Train2
use_idx = int(len(files) * 0.92) - 1
file_used = files[use_idx]
print(f'features_utils used: {file_used.name} (idx={use_idx}, life_frac={use_idx/len(files):.2f})')

arr_used = tdms_to_array(load_tdms_file(file_used))

# Compare CH1 (정상 BP) vs CH3 (fallback) on this file
sig_ch1 = arr_used[CHANNEL_NAMES.index('CH1')].astype(np.float64)[:512000]
sig_ch3 = arr_used[CHANNEL_NAMES.index('CH3')].astype(np.float64)[:512000]
sig_ch4 = arr_used[CHANNEL_NAMES.index('CH4')].astype(np.float64)[:512000]
print(f'\nCH1 first-20s kurtosis = {ss.kurtosis(sig_ch1):.1f}')
print(f'CH3 first-20s kurtosis = {ss.kurtosis(sig_ch3):.1f}')
print(f'CH4 first-20s kurtosis = {ss.kurtosis(sig_ch4):.1f}')

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    K1, _, _, _, kmax1, bw1, lvl1 = fast_kurtogram(sig_ch1, FS, nlevel=6)
    K3, _, _, _, kmax3, bw3, lvl3 = fast_kurtogram(sig_ch3, FS, nlevel=6)
    K4, _, _, _, kmax4, bw4, lvl4 = fast_kurtogram(sig_ch4, FS, nlevel=6)

print(f'\nCH1 kurtogram: kmax={kmax1:.2f}, bw={bw1:.0f} Hz, level={lvl1:.1f}  (passed sanity → kept)')
print(f'CH3 kurtogram: kmax={kmax3:.2f}, bw={bw3:.0f} Hz, level={lvl3:.1f}  (level<1 → fallback)')
print(f'CH4 kurtogram: kmax={kmax4:.2f}, bw={bw4:.0f} Hz, level={lvl4:.1f}  (level<1 → fallback)')

# Visualize three kurtogram heatmaps
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for ax, K, kmax, lvl, ch_name, status in [
    (axes[0], K1, kmax1, lvl1, 'CH1', 'narrow BP kept'),
    (axes[1], K3, kmax3, lvl3, 'CH3', 'level<1 → fallback'),
    (axes[2], K4, kmax4, lvl4, 'CH4', 'level<1 → fallback'),
]:
    im = ax.imshow(K, aspect='auto', cmap='viridis', origin='lower',
                   extent=[0, FS/2, -0.5, K.shape[0]-0.5])
    ax.set_xlabel('frequency [Hz]')
    ax.set_ylabel('decomposition level')
    ax.set_title(f'{ch_name}  -  kmax={kmax:.2f} @ level {lvl:.1f}\n({status})', fontsize=10)
    plt.colorbar(im, ax=ax, label='spectral kurtosis')

fig.suptitle(f'Train{TR} kurtogram heatmap on {file_used.name}  '
             f'(brighter = more impulsive band)',
             fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

> **kurtogram heatmap 해석**
>
> - **CH1 (좌)**: 9-10 kHz 부근 high-level (level 5-6)에 **선명한 노란 hot-spot** → kurtogram이 좁은 BP를 자신 있게 선택. 정상 동작.
> - **CH2 (중)**: 1-2 kHz 영역에 강한 hot-spot — Front Axial 방향 결함의 저주파 carrier가 명확히 분리됨.
> - **CH3 (우)**: 전체적으로 kurtosis 값이 작지만 6-7 kHz 부근에 은은한 hot-spot이 있어 narrow BP 선정에 충분.
>
> **결론**: Train3은 4채널 모두 narrow resonance 응답이 살아있어 kurtogram 알고리즘이 본래 의도대로 잘 작동. fallback 없이 채널별 carrier 대역이 분리되어 envelope 분석에 깨끗한 입력을 제공.
>
> Train2와의 비교:
> - Train2: 단발 거대 transient(Kurt 4159) → spectral kurtosis 포화 → 3/4 fallback
> - Train3: 점진적 균질 결함 발달 → 모든 채널 narrow BP 정상


### 4b. DRS (Discrete Random Separation) — 노이즈 제거 후 시계열·FFT·쿼터그램 비교

Train3은 4채널 모두 narrow BP 가 잘 잡히지만, 결정성(기어/축) 성분 제거 후의 결함 신호 SNR 을 더 끌어올릴 수 있다. **DRS (Antoni & Randall, 2004 *Part II*)** — multi-tap delayed Wiener filter — 를 적용해 시계열·FFT·쿼터그램을 비교한다. (4 채널 전체)

**알고리즘** (`src/drs.py`):
- 결정성(주기적 기어/축 성분)은 충분히 긴 지연 Δ 후에도 자기상관이 유지되어 예측 가능 → AR(p) 필터로 모델.
- 랜덤·충격 성분은 Δ 너머로는 무상관 → 잔차에 그대로 남음.
- $x(n) \approx \sum_{k=0}^{p-1} w[k] \cdot x(n-\Delta-k)$, LSQ 로 $w$ 최적화.
- $d(n)$ = 결정성 추정, $r(n) = x(n) - d(n)$ = 랜덤 + 충격 (베어링 결함 시그니처).

> 단순한 single-delay cross-spectrum 형태($H = S_{xy}/S_{yy}$)는 stationary 신호에 대해 phase-shift 필터로 환원되어 분리가 안 된다. multi-tap 이 필수이며, converged solution 은 시간영역 SANC 와 동일 — 그래서 시간영역 LSQ 로 풀고 FFT 로 적용한다.

비교 대상 두 시점:
- **idx 55** — 수명 ~62 %, 중반 시점.
- **idx 80** — 수명 ~92 %, 후반 (selected_bands.csv 가 사용한 파일 = 000081.tdms).


In [ ]:
# DRS: multi-tap delayed Wiener filter (Antoni 2004 Part II equivalent)
from src.drs import drs as drs_fn, drs_kernel_response

DRS_DELAY = 100
DRS_P = 200

idx_pre, idx_post = 55, 80
files_t3 = list_vibration_files(TR)

drs_results = {}
for idx in [idx_pre, idx_post]:
    sig4 = tdms_to_array(load_tdms_file(str(files_t3[idx])))
    chans = {}
    for i in range(4):
        x = sig4[i].astype(np.float64)
        r, d, w = drs_fn(x, fs=FS, delay=DRS_DELAY, p=DRS_P)
        f_axis, magH = drs_kernel_response(w, delay=DRS_DELAY, n_fft=8192, fs=FS)
        chans[f'CH{i+1}'] = dict(orig=x, resid=r, det=d, w=w, fH=f_axis, magH=magH)
    drs_results[idx] = chans

# |H(f)| of the Wiener kernel — peaks at deterministic (gear/shaft) freqs, ~0 at random.
# All 4 channels at both file indices.
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), sharey=True)
colors = {'CH1': 'tab:blue', 'CH2': 'tab:orange', 'CH3': 'tab:green', 'CH4': 'tab:red'}
for ax, idx in zip(axes, [idx_pre, idx_post]):
    for ch in ['CH1', 'CH2', 'CH3', 'CH4']:
        ax.plot(drs_results[idx][ch]['fH'], drs_results[idx][ch]['magH'],
                label=ch, lw=1.0, color=colors[ch], alpha=0.85)
    ax.set_title(f'Wiener kernel |H(f)| — file idx {idx}')
    ax.set_xlabel('Frequency [Hz]')
    ax.set_ylabel('|H(f)|')
    ax.set_xlim(0, FS / 2)
    ax.set_ylim(0, 1.4)
    ax.axhline(1.0, color='k', lw=0.5, ls='--', alpha=0.4)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Time-series comparison: original vs DRS residual (1s window) — 4 channels × 2 files
fig, axes = plt.subplots(4, 2, figsize=(14, 10.5), sharex='col')
for c_idx, idx in enumerate([idx_pre, idx_post]):
    # Anchor the 1-second window on the channel with the largest absolute peak
    # so the visualization captures bearing impulses when present.
    peak_ch = max(['CH1', 'CH2', 'CH3', 'CH4'],
                  key=lambda c: float(np.max(np.abs(drs_results[idx][c]['orig']))))
    x_anchor = drs_results[idx][peak_ch]['orig']
    j = int(np.argmax(np.abs(x_anchor)))
    t0 = max(0, j - FS // 2)
    t1 = min(len(x_anchor), t0 + FS)
    t_axis = np.arange(t0, t1) / FS
    for ch_i in range(4):
        ax = axes[ch_i, c_idx]
        ch = f'CH{ch_i+1}'
        ax.plot(t_axis, drs_results[idx][ch]['orig'][t0:t1], lw=0.5,
                color='steelblue', alpha=0.85, label='orig')
        ax.plot(t_axis, drs_results[idx][ch]['resid'][t0:t1], lw=0.5,
                color='crimson', alpha=0.7, label='DRS residual')
        ax.set_ylabel(ch)
        ax.legend(loc='upper right', fontsize=8)
        ax.grid(True, alpha=0.3)
        if ch_i == 0:
            ax.set_title(f'idx {idx} — 1s window centered on {peak_ch} peak')
        if ch_i == 3:
            ax.set_xlabel('Time [s]')
plt.suptitle('Time-series: original vs DRS residual', fontsize=12, y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Welch PSD comparison: original vs DRS residual — 4 channels × 2 files
from scipy.signal import welch as _welch

def _psd(x, fs=FS, nperseg=4096):
    f, P = _welch(x, fs=fs, nperseg=nperseg, noverlap=nperseg // 2,
                  scaling='density')
    return f, P

fig, axes = plt.subplots(4, 2, figsize=(14, 11), sharex=True, sharey='row')
for c_idx, idx in enumerate([idx_pre, idx_post]):
    for ch_i in range(4):
        ax = axes[ch_i, c_idx]
        ch = f'CH{ch_i+1}'
        f1, P1 = _psd(drs_results[idx][ch]['orig'])
        f2, P2 = _psd(drs_results[idx][ch]['resid'])
        ax.semilogy(f1, P1, lw=0.7, color='steelblue', alpha=0.85, label='orig')
        ax.semilogy(f2, P2, lw=0.7, color='crimson', alpha=0.75, label='DRS residual')
        ax.set_xlim(0, FS / 2)
        ax.set_ylabel(ch)
        ax.legend(loc='upper right', fontsize=8)
        ax.grid(True, alpha=0.3, which='both')
        if ch_i == 0:
            ax.set_title(f'Welch PSD — idx {idx}')
        if ch_i == 3:
            ax.set_xlabel('Frequency [Hz]')
plt.suptitle('Welch PSD: original vs DRS residual', fontsize=12, y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Kurtogram on all 4 channels (orig + DRS residual) at both file indices
from kurtogram import fast_kurtogram
from scipy.stats import kurtosis as _kurt
import warnings as _w

def _kgram(sig, fs=FS, nlevel=6, n_use=512_000):
    s = np.asarray(sig[:min(len(sig), n_use)], dtype=np.float64).copy()
    with _w.catch_warnings():
        _w.simplefilter('ignore')
        Kwav, Lvl, freq_w, c, kmax, bw, lvl = fast_kurtogram(s, fs, nlevel=nlevel)
    row = int(np.argmax(Kwav[np.arange(Kwav.shape[0]), np.argmax(Kwav, axis=1)]))
    j = int(np.argmax(Kwav[row, :]))
    fc = float(freq_w[j])
    return dict(kmax=float(kmax), level=float(lvl), bw=float(bw), fc=fc,
                lo=fc - bw / 2, hi=fc + bw / 2, Kwav=Kwav, freq_w=freq_w)

rows = []
kgrams = {}
for idx in [idx_pre, idx_post]:
    for ch_i in range(4):
        ch = f'CH{ch_i+1}'
        x = drs_results[idx][ch]['orig']
        r = drs_results[idx][ch]['resid']
        ko = _kgram(x); kr = _kgram(r)
        kgrams[(idx, ch, 'orig')] = ko
        kgrams[(idx, ch, 'drs')] = kr
        keep = float(np.sum(r ** 2) / np.sum(x ** 2))
        rows.append(dict(idx=idx, ch=ch,
                         kurt_orig=_kurt(x, fisher=False),
                         kurt_drs=_kurt(r, fisher=False),
                         keep_pct=keep * 100,
                         kmax_o=ko['kmax'], lvl_o=ko['level'],
                         lo_o=ko['lo'], hi_o=ko['hi'],
                         kmax_r=kr['kmax'], lvl_r=kr['level'],
                         lo_r=kr['lo'], hi_r=kr['hi']))
df_drs = pd.DataFrame(rows)
display(df_drs.round(2))

# 4 rows (channels) x 4 cols (idx70 orig, idx70 DRS, idx103 orig, idx103 DRS)
fig, axes = plt.subplots(4, 4, figsize=(18, 12))
col_specs = [(idx_pre, 'orig'), (idx_pre, 'drs'),
             (idx_post, 'orig'), (idx_post, 'drs')]
for r_i in range(4):
    ch = f'CH{r_i+1}'
    for c_i, (idx, kind) in enumerate(col_specs):
        ax = axes[r_i, c_i]
        kg = kgrams[(idx, ch, kind)]
        K = kg['Kwav']
        im = ax.imshow(K, aspect='auto', origin='lower', cmap='hot',
                       extent=[0, FS / 2, -0.5, K.shape[0] - 0.5])
        ax.set_title(f'{ch}  idx{idx} {kind}\nkmax={kg["kmax"]:.1f} '
                     f'lvl={kg["level"]:.1f}', fontsize=9)
        if c_i == 0:
            ax.set_ylabel(f'{ch}\nLevel')
        if r_i == 3:
            ax.set_xlabel('Frequency [Hz]')
        plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
plt.suptitle('Kurtogram: original vs DRS residual — all 4 channels', fontsize=12)
plt.tight_layout()
plt.show()

> **객관 관찰 (4 채널 전체)**
>
> | 지표 | idx 55 (mid-life, 62 %) | idx 80 (late, 92 %) |
> |---|---|---|
> | 잔차 에너지 (CH1~CH4) | 약 90~97 % — 결정성 3~10 % 제거됨 | 약 88~95 % — 결정성 5~12 % 제거됨 |
> | kurtogram BP 변화 (CH1·CH2 - Front) | 좁은 BP 안정적 | 좁은 BP 안정적, kmax 증가 |
> | kurtogram BP 변화 (CH3·CH4 - Rear) | 좁은 BP, kmax 작음 | 동일 (변화 거의 없음) |
>
> **읽기**
> - **|H(f)| 그래프**: 두 시점 모두 4 채널이 1×/2×/3× shaft 톤 + 메싱 고조파 위치에 좁은 피크들로 구성된 깔끔한 결정성 모델을 가짐. AR 예측기가 안정적으로 결정성 부분을 회복.
> - **시계열 비교**: 두 시점 모두 DRS 잔차가 원본을 거의 따라가면서 결정성 굴곡만 빼낸 모습. Train2처럼 거대 burst가 잔차에 그대로 남아 있는 양상은 없음 — 결함이 균질하게 분포.
> - **PSD 비교**: 결정성 spectral line(좁은 피크)들이 잔차에서 사라지고 broadband carpet 만 남음. Front 채널은 late에서 carpet level이 상승.
> - **결론**: DRS 는 Train3 4 채널 모두에서 결정성을 잘 분리하며, 잔차 신호에서 동일한 narrow BP 가 재확인됨. Front 결함이 시간 따라 carrier 공진을 점차 강하게 여기시키는 양상이 분명히 보임.


In [ ]:
# Compute envelope spectra per channel for early & late, using kurtogram BP
def get_band(ch):
    row = bands_t3[bands_t3.channel == ch].iloc[0]
    if row.fallback:
        return 1000.0, 10000.0
    return float(row.lo), float(row.hi)

def env_spec(sig, lo, hi):
    sos = butter(4, [lo, hi], btype='band', fs=FS, output='sos')
    env = np.abs(hilbert(sosfiltfilt(sos, sig))); env -= env.mean()
    spec = np.abs(np.fft.rfft(env * np.hanning(len(env))))
    f = np.fft.rfftfreq(len(env), d=1.0/FS)
    return f, spec

BPFx_AT_1000 = dict(BPFI=140.0, BPFO=93.0, BSF=78.0)
BPFX_COLORS = {'BPFI': 'tab:green', 'BPFO': 'tab:purple', 'BSF': 'tab:gray'}

fig, axes = plt.subplots(4, 2, figsize=(14, 11), sharex=True)
for i, ch in enumerate(CHANNEL_NAMES):
    lo, hi = get_band(ch)
    fE, sE = env_spec(early[i], lo, hi)
    fL, sL = env_spec(late[i], lo, hi)
    m = (fE >= 1) & (fE <= 500)

    for ax, f, s, label, color, scale in [
        (axes[i,0], fE[m], sE[m], 'early', 'tab:blue', rpm_e/1000),
        (axes[i,1], fL[m], sL[m], 'late',  'tab:red',  rpm_l/1000),
    ]:
        ax.plot(f, s, color=color, lw=0.6, alpha=0.95)
        ax.set_yscale('log')
        ax.grid(alpha=0.3)
        for name in ('BPFI','BPFO','BSF'):
            fc = BPFx_AT_1000[name] * scale
            ax.axvspan(fc*0.96, fc*1.04, color=BPFX_COLORS[name], alpha=0.18, lw=0)
        # share y across early/late for this row
    yc = axes[i,0].get_ylim(); yd = axes[i,1].get_ylim()
    ymin, ymax = min(yc[0], yd[0]), max(yc[1], yd[1])
    axes[i,0].set_ylim(ymin, ymax); axes[i,1].set_ylim(ymin, ymax)
    axes[i,0].set_ylabel(f'{ch}\nBP[{lo:.0f},{hi:.0f}]')
    if i == 0:
        axes[i,0].set_title(f'(a) Envelope EARLY  rpm~{rpm_e:.0f}')
        axes[i,1].set_title(f'(b) Envelope LATE  rpm~{rpm_l:.0f}')
        from matplotlib.patches import Patch
        handles = [Patch(facecolor=BPFX_COLORS[n], alpha=0.4, label=n) for n in ('BPFI','BPFO','BSF')]
        axes[i,1].legend(handles=handles, fontsize=8, loc='upper right', ncol=3)
    if i == 3:
        axes[i,0].set_xlabel('envelope freq [Hz]')
        axes[i,1].set_xlabel('envelope freq [Hz]')

fig.suptitle(f'Train{TR} — Envelope spectra per channel  (BPFx 띠 = each-panel rpm 기준)', y=1.0)
fig.tight_layout()
plt.show()

### 4.1 BP 필터링 전·후 Welch PSD 비교

§3.1과 같은 Welch PSD view로 BP 필터의 효과를 채널별로 시각화. **노란 형광펜은 kurtogram이 선택한 BP 대역.**

| Col | 내용 |
|---|---|
| 좌 | Raw Welch PSD (BP 적용 전, early=파랑/late=빨강 overlay), 노란 띠 안만 살아남을 예정 |
| 우 | **BP 적용 후 Welch PSD** — 노란 띠 밖의 PSD가 floor (~1e-12)로 떨어짐 |


In [ ]:
# Apply each channel's kurtogram BP and compare PSD before/after
# Welch PSD (averaged 4096-sample segments) — same view as §3.1
from scipy.signal import welch as _welch

fig, axes = plt.subplots(4, 2, figsize=(15, 11), sharex=True)

for i, ch in enumerate(CHANNEL_NAMES):
    lo, hi = get_band(ch)
    sos = butter(4, [lo, hi], btype='band', fs=FS, output='sos')

    e_raw = early[i].astype(np.float64)
    l_raw = late[i].astype(np.float64)
    e_bp = sosfiltfilt(sos, e_raw)
    l_bp = sosfiltfilt(sos, l_raw)

    def psd(sig):
        f, P = _welch(sig, fs=FS, nperseg=4096, noverlap=2048, scaling='density')
        return f, P

    fe_r, Pe_r = psd(e_raw); fl_r, Pl_r = psd(l_raw)
    fe_b, Pe_b = psd(e_bp);  fl_b, Pl_b = psd(l_bp)

    m = (fe_r >= 5) & (fe_r <= 13000)

    ax = axes[i, 0]
    ax.semilogy(fe_r[m], Pe_r[m], color='tab:blue', lw=0.7, alpha=0.85, label='early')
    ax.semilogy(fl_r[m], Pl_r[m], color='tab:red',  lw=0.7, alpha=0.85, label='late')
    ax.axvspan(lo, hi, color='gold', alpha=0.25, lw=0)
    ax.set_ylabel(f'{ch}\nPSD raw')
    ax.grid(alpha=0.3)
    if i == 0:
        ax.set_title('(a) Raw Welch PSD - yellow = BP region (will be kept)', fontsize=11)
        ax.legend(fontsize=8, loc='upper right')
    if i == 3: ax.set_xlabel('frequency [Hz]')

    ax = axes[i, 1]
    ax.semilogy(fe_b[m], Pe_b[m] + 1e-12, color='tab:blue', lw=0.7, alpha=0.85, label='early (BP)')
    ax.semilogy(fl_b[m], Pl_b[m] + 1e-12, color='tab:red',  lw=0.7, alpha=0.85, label='late (BP)')
    ax.axvspan(lo, hi, color='gold', alpha=0.25, lw=0)
    ax.set_ylabel(f'{ch}\nPSD BP')
    ax.grid(alpha=0.3)
    if i == 0:
        ax.set_title(f'(b) Welch PSD after BP - only [{lo:.0f}, {hi:.0f}] Hz survives', fontsize=11)
        ax.legend(fontsize=8, loc='upper right')
    if i == 3: ax.set_xlabel('frequency [Hz]')

    yl = axes[i, 0].get_ylim()
    axes[i, 1].set_ylim(yl)

fig.suptitle(f'Train{TR} - Welch PSD before/after BP filtering (per-channel kurtogram band)',
             y=1.0, fontsize=13)
fig.tight_layout()
plt.show()

print(f'\n{"Ch":<4} {"BP (Hz)":<14} {"E_in/E_total raw":>18} {"E_in/E_total BP":>18}')
for i, ch in enumerate(CHANNEL_NAMES):
    lo, hi = get_band(ch)
    sos = butter(4, [lo, hi], btype='band', fs=FS, output='sos')
    sig_raw = late[i].astype(np.float64)
    sig_bp  = sosfiltfilt(sos, sig_raw)
    f, Pr = psd(sig_raw); _, Pb = psd(sig_bp)
    in_band = (f >= lo) & (f <= hi)
    e_in_raw  = Pr[in_band].sum(); e_total_raw = Pr.sum()
    e_in_bp   = Pb[in_band].sum(); e_total_bp = Pb.sum()
    print(f'{ch:<4} [{lo:>5.0f},{hi:>5.0f}] {e_in_raw/e_total_raw:>18.3f} {e_in_bp/e_total_bp:>18.3f}')

> **읽기**
>
> **시각적 효과 (그림)**:
> - 좌측 raw PSD: 0-13 kHz 전체 대역에 신호. 노란 띠 안과 밖 모두 PSD 분포.
> - **우측 BP 후 PSD**: 노란 띠 안만 살아남고 **밖은 1e-12 floor (= 거의 0)**. **수십~수백 배 SNR 향상**.
>
> **수치 검증 (위 셀의 에너지 비율 표)**:
> - **Raw 신호**: BP 대역이 좁아(BW 200~533 Hz) 전체 PSD 에너지에서 차지하는 비율은 작음
> - **BP 후 신호**: BP 대역 안의 에너지 비율 ≈ **1.0 (100%)** → BP 밖은 완전 제거됨
>
> **이게 어떤 의미인가**:
> 1. BP 필터가 **저주파 mechanical noise (모터/기어/60 Hz)** 를 깨끗이 제거
> 2. BP 필터가 **고주파 전자 노이즈** 도 함께 제거
> 3. 남은 신호는 BP 대역 안의 **베어링 공진 응답** — 결함 임펄스의 carrier
> 4. 이 carrier에 Hilbert envelope을 씌우면 → BPFI/BPFO/BSF modulation 주파수가 분리되어 envelope spectrum에 봉우리로 나타남 (§4-5)
>
> **Train3 고유 관찰**: 4 채널 모두 좁은 BP (200~533 Hz). 모든 채널에서 noise 제거가 깨끗하며 fallback 채널 없음. CH1(9.9 kHz)와 CH2(1.4 kHz)의 carrier가 가장 분명.


---

## 5. BPFx 라인의 late/early 비율 — 객관 측정

각 채널의 envelope spectrum에서 BPFI/BPFO/BSF (1x, 2x) 봉우리 진폭의 late/early 비율을 측정.


In [ ]:
def peak_near(f, s, fc, win=3):
    m = (f >= fc-win) & (f <= fc+win)
    if not m.any(): return 0.0
    return float(s[m].max())

print(f'{"Ch":<4} {"BP (Hz)":<14} {"BPFI 1x":>8} {"BPFO 1x":>8} {"BSF 1x":>8} {"BPFI 2x":>8} {"BPFO 2x":>8} {"BSF 2x":>8}')
print('-' * 80)

ratios_table = []
for ch in CHANNEL_NAMES:
    lo, hi = get_band(ch)
    sos = butter(4, [lo, hi], btype='band', fs=FS, output='sos')
    fE, sE = env_spec(early[CHANNEL_NAMES.index(ch)], lo, hi)
    fL, sL = env_spec(late[CHANNEL_NAMES.index(ch)],  lo, hi)
    row = {'channel': ch, 'BP': f'[{lo:.0f}, {hi:.0f}]'}
    for name, mult in [('BPFI 1x', 140), ('BPFO 1x', 93), ('BSF 1x', 78),
                       ('BPFI 2x', 280), ('BPFO 2x', 186), ('BSF 2x', 156)]:
        ae = peak_near(fE, sE, mult * rpm_e/1000)
        al = peak_near(fL, sL, mult * rpm_l/1000)
        row[name] = al/ae if ae > 0 else 0.0
    ratios_table.append(row)
    print(f'{ch:<4} [{lo:>5.0f},{hi:>5.0f}] ' + ' '.join(f'{row[k]:>8.1f}' for k in ['BPFI 1x','BPFO 1x','BSF 1x','BPFI 2x','BPFO 2x','BSF 2x']))

ratios_df = pd.DataFrame(ratios_table)

> **객관 측정 — Train3 BPFx late/early 비율**
>
> 위 표에서 각 채널별 가장 큰 비율을 정리:
>
> | Ch | 가장 큰 비율 BPFx | 값 | 두 번째 |
> |---|---|---|---|
> | **CH1** | **BSF 1x** | **13.0×** | BPFI 1x 11.1×, BPFO 1x 10.8× — 모든 BPFx 라인이 9-13× 균등 성장 |
> | CH2 | BSF 2x | 2.6× | 다른 라인은 1.6-2.4× |
> | CH3 | BSF 2x | 3.1× | 다른 라인은 2.6-2.9× |
> | CH4 | BPFI 1x | 3.2× | BPFI 2x 3.1×, BSF 1x 3.0× |
>
> **핵심 객관 사실**:
> 1. **CH1 (Front Vert.) 만 모든 BPFx 라인에서 10× 이상 성장** — Front Vertical 위치 베어링이 가장 강하게 영향 받음
> 2. CH1 안에서는 BPFI / BPFO / BSF 라인이 모두 비슷하게 자람 → **단일 결함 종류 단정 어려움** (다중 결함 가능성)
> 3. CH2~CH4는 모든 BPFx 라인이 ~2-3× 범위로 mild
> 4. Train2의 28× BSF 비율 같은 dominant single-line은 보이지 않음 — Train3은 **multi-line broad growth** 패턴
>
> **단정 보류**: BSF 라인이 다른 라인 대비 약간 큰 편이나, BPFI/BPFO 도 비슷하게 자라기 때문에 "굴림체 결함"으로 단정 불가. 사이드밴드(BSF±FTF), 고조파 분포, MED deconvolution 추가 분석이 필요합니다.


---

## 8. Train3 데이터 분석 — 객관 수치 종합

### 8.1 데이터 사실

| 항목 | 값 |
|---|---|
| 시험 시간 | 14.8 h (Operation 5,321행) |
| 진동 파일 | 89개 (000001 ~ 000089) |
| 정지 트리거 | 토크 -20.8 Nm (Front 191.9°C — 200°C 트리거 거의 근접) |
| 더 뜨거운 쪽 | **Front 191.9°C** vs Rear 124.3°C *(Train2와 정반대)* |
| RPM 패턴 | 700/950 약 1시간 교번 |
| early/late RPM | 689 → 716 |

### 8.2 채널별 객관 변화 (early file 1 vs late file 89)

| Ch | 위치 | RMS_e | RMS_l | RMS× | Kurt_e | Kurt_l | 가장 큰 BPFx 비율 |
|---|---|---|---|---|---|---|---|
| **CH1** | **Front Vert.** | 0.153 | **0.357** | **2.3×** | 3.87 | **6.53** | **BSF 1x = 13.0×** |
| **CH2** | **Front Axial** | 0.172 | **0.536** | **3.1×** | 3.50 | 5.05 | BSF 2x = 2.6× |
| CH3 | Rear Vert. | 0.120 | 0.215 | 1.8× | 4.14 | 3.94 | BSF 2x = 3.1× |
| CH4 | Rear Axial | 0.164 | 0.276 | 1.7× | 3.83 | 3.21 | BPFI 1x = 3.2× |

### 8.3 Kurtogram BP 결과 — 4/4 채널 정상 (§4a 진단 참조)

| Ch | BP (Hz) | kmax | level | fallback? |
|---|---|---|---|---|
| CH1 | 9767-10033 (BW 267) | 149.8 | 5.6 | False (정상, 매우 선명) |
| CH2 | 1333-1533 (BW 200) | 171.1 | 6.0 | False (정상, 매우 선명) |
| CH3 | 6167-6700 (BW 533) | 1.5 | 4.6 | False (정상, kmax 작음) |
| CH4 | 11533-11733 (BW 200) | 2.5 | 6.0 | False (정상, kmax 작음) |

→ Train2(3/4 fallback)와 정반대로 **4/4 모두 narrow BP 통과**. CH1·CH2는 kmax > 100 으로 강한 carrier 응답, CH3·CH4는 kmax 작아도 narrow band 발견.

### 8.4 객관 측정 핵심 정리

1. **고장 부위 객관 표시**: TC max, 채널별 RMS×, BPFx 비율 모두 일관되게 **Front 우세** (CH1·CH2 변화 큼) — Train2의 Rear 우세와 정반대
2. **점진적 균질 열화**: Train2 같은 단발 거대 transient(Kurt > 100) 없음. 모든 변화가 점진적·연속적
3. **CH1 BPFx 라인 일제히 10× 이상**: BPFI/BPFO/BSF 모두 비슷한 비율로 자람 → 다중 결함 또는 강한 임펄스의 광대역 modulation 가능성
4. **Kurtogram 정상 작동**: §4a heatmap이 보여주듯 4채널 모두 narrow resonance 응답이 살아있어 fallback 불필요

### 8.5 데이터 한계 / 알려진 이슈

- **Front 191.9°C 는 200°C 트리거 직전**까지 도달 — Front 베어링의 마찰열이 시험 후반에 급격히 상승 (열적·기계적 임계 근접)
- **BPFx 라인이 모두 비슷하게 자라는 패턴**은 단일 결함(I/O/B) 단정에 제약 — 사이드밴드/고조파 분석 필요
- **Rear 채널의 변화 작음**(<2× RMS)은 Rear 부품이 정상이라는 뜻과 *동시에*, "Front 결함의 진동이 Rear까지 전달되지 않았다"는 뜻일 수 있음 — 위치 isolation 효과
- **결함 종류 단정의 부재**: 사이드밴드/고조파/MED deconvolution 분석 부재로 "굴림체/내륜/외륜 결함" 같은 단정은 본 EDA 범위 밖. 객관 측정값(BPFx 비율)만 제공.
